# Agent Memory Systems

Memory is what separates a stateless Q&A system from an agent that learns and improves over time. Modern agent memory systems combine multiple storage types to handle different temporal horizons and access patterns.

## Memory Taxonomy

Cognitive science distinguishes three memory types, and agent architectures map onto them:

**Episodic memory**: specific past events — 'Last Tuesday the user asked about Q4 projections.' Implemented as conversation history (in-context) or a timestamped event log (external store).

**Semantic memory**: general world knowledge — 'The user works at Acme Corp and prefers concise answers.' Implemented as a knowledge base or vector store of user/world facts.

**Procedural memory**: how to perform tasks — 'When the user asks for a report, use this template and format.' Implemented as retrieved few-shot examples or fine-tuned behavior.

Different tasks require different memory types. A customer service agent needs semantic memory (customer history). A coding assistant needs procedural memory (how this codebase is structured). A personal assistant needs episodic memory (what was discussed before).

In [ ]:
from dataclasses import dataclass, field
from typing import Callable
import time, json, math

@dataclass
class MemoryEntry:
    content: str
    memory_type: str
    timestamp: float
    importance: float = 1.0
    embedding: list = field(default_factory=list)
    access_count: int = 0

class InContextMemory:
    def __init__(self, max_tokens: int = 2000):
        self.entries: list = []
        self.max_tokens = max_tokens

    def add(self, content: str, memory_type: str = 'episodic', importance: float = 1.0):
        entry = MemoryEntry(content=content, memory_type=memory_type,
                             timestamp=time.time(), importance=importance)
        self.entries.append(entry)
        self._evict_if_needed()

    def _evict_if_needed(self):
        total = sum(len(e.content.split()) for e in self.entries)
        while total > self.max_tokens and self.entries:
            # Evict oldest, lowest-importance entry
            to_evict = min(self.entries, key=lambda e: e.importance * 0.3 + (time.time()-e.timestamp) * -0.001)
            self.entries.remove(to_evict)
            total = sum(len(e.content.split()) for e in self.entries)

    def retrieve(self, query: str, k: int = 5) -> list:
        # Simple keyword overlap for retrieval
        query_words = set(query.lower().split())
        scored = []
        for e in self.entries:
            e_words = set(e.content.lower().split())
            overlap = len(query_words & e_words)
            score = overlap * e.importance
            if score > 0:
                scored.append((score, e))
        scored.sort(key=lambda x: -x[0])
        results = [e for _, e in scored[:k]]
        for e in results:
            e.access_count += 1
        return results

    def render(self) -> str:
        if not self.entries:
            return '[empty memory]'
        lines = []
        for e in self.entries:
            lines.append(f'[{e.memory_type}] {e.content}')
        return '\n'.join(lines)

mem = InContextMemory(max_tokens=500)
mem.add('User is a data scientist at Acme Corp.', 'semantic', importance=2.0)
mem.add('User prefers concise bullet-point answers.', 'semantic', importance=2.0)
mem.add('On 2024-01-15, user asked about PyTorch optimizers.', 'episodic', importance=1.0)
mem.add('On 2024-01-16, user asked for help debugging a CUDA error.', 'episodic', importance=1.0)
mem.add('Common task: generate Python code with type hints.', 'procedural', importance=1.5)

print('All memory:')
print(mem.render())
print('\nRetrieved for query: "python debugging help":')
for e in mem.retrieve('python debugging help', k=3):
    print(f'  [{e.memory_type}] {e.content}')

## Vector Store Memory

For agents that accumulate many memories over time, keyword retrieval breaks down. Vector store memory uses embeddings to find semantically similar memories regardless of exact word overlap.

In [ ]:
class SimpleVectorMemory:
    def __init__(self, embed_fn: Callable):
        self.embed = embed_fn
        self.entries: list = []

    def add(self, content: str, memory_type: str = 'episodic', importance: float = 1.0):
        emb = self.embed(content)
        entry = MemoryEntry(content=content, memory_type=memory_type,
                             timestamp=time.time(), importance=importance, embedding=emb)
        self.entries.append(entry)

    def _cosine_sim(self, a: list, b: list) -> float:
        dot = sum(x*y for x, y in zip(a, b))
        norm_a = math.sqrt(sum(x*x for x in a))
        norm_b = math.sqrt(sum(x*x for x in b))
        return dot / (norm_a * norm_b + 1e-8)

    def retrieve(self, query: str, k: int = 3) -> list:
        q_emb = self.embed(query)
        scored = [(self._cosine_sim(q_emb, e.embedding), e) for e in self.entries]
        scored.sort(key=lambda x: -x[0])
        return [e for _, e in scored[:k]]

import random
rng = random.Random(42)
def mock_embed(text: str) -> list:
    # Hash-based pseudo-embedding (256 dims)
    rng2 = random.Random(hash(text[:50]) % 100000)
    return [rng2.gauss(0, 1) for _ in range(32)]

vmem = SimpleVectorMemory(mock_embed)
vmem.add('User asked about transformer attention mechanisms', 'episodic')
vmem.add('User prefers concise answers with code examples', 'semantic', 2.0)
vmem.add('Project uses PyTorch and CUDA 12.1', 'semantic', 2.0)
vmem.add('Previous bug: indexing error in batch processing loop', 'episodic')
vmem.add('User asked about learning rate scheduling', 'episodic')

results = vmem.retrieve('help with neural network training', k=3)
print('Vector memory retrieval for: "help with neural network training"')
for e in results:
    print(f'  [{e.memory_type}] {e.content}')

## Summary Memory and Long-Term Compression

For very long agent sessions, even a vector store becomes expensive to query across thousands of entries. Summary memory periodically compresses older entries:

1. Take the oldest N entries
2. Ask the model to summarize them into key facts
3. Replace those entries with the summary
4. Repeat as the agent accumulates more memories

This mirrors how humans compress episodic memory into semantic memory over time — specific episodes fade; the general lessons remain.